# MJLab (MuJoCo Warp) + rl_games — Colab training

Trains a **Unitree Go1 flat-terrain velocity policy** end-to-end on the Colab GPU:
GPU-parallel simulation (MuJoCo Warp), GPU PPO (rl_games), no CPU round-trips.

**Runtime: L4 or A100 recommended** (Runtime → Change runtime type). ~6-8 min of
training on L4-class hardware at 2048 envs; A100 runs 4096. T4 is untested.

Companion notebook: `mjlab_visualization_colab.ipynb` (render video + velocity probe) —
run it in the same session to reuse this checkpoint.

In [ ]:
# rl_games from git until the 2.0 PyPI release (then: %pip install rl-games mjlab)
RL_GAMES_REF = 'VM/feature/b1-ppo'   # flip to 'master' after the branch merges, PyPI after release

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    f'git+https://github.com/Denys88/rl_games.git@{RL_GAMES_REF}',
    'mjlab==1.5.0', 'warp-lang==1.14.0', 'mujoco-warp==3.10.0.1',
    'imageio[ffmpeg]', 'tensorboard'])
# warp pins: warp-lang 1.15 / mujoco-warp 3.10.0.2 currently crash mjlab's reset
# (CUDA illegal memory access in the warp<->torch mask interop) -- verified 2026-07-18.
# Drop the pins once fixed upstream.

import torch
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {gpu} ({vram:.0f} GiB)")
if 'T4' in gpu:
    print("WARNING: T4 is untested for MuJoCo Warp and lacks bf16 support -- expect issues and "
          "slow training. Use an L4 or A100 runtime (Colab Pro) for the intended experience.")
NUM_ENVS = 4096 if vram >= 40 else (2048 if vram >= 20 else 1024)
print("NUM_ENVS =", NUM_ENVS)

In [ ]:
# Fetch the shipped Go1 config from the repo (the pip package ships code, not configs)
import urllib.request
CFG_URL = (f'https://raw.githubusercontent.com/Denys88/rl_games/{RL_GAMES_REF}'
           '/rl_games/configs/mjlab/ppo_go1_velocity.yaml')
urllib.request.urlretrieve(CFG_URL, 'ppo_go1_velocity.yaml')
print('fetched ppo_go1_velocity.yaml')

In [ ]:
import yaml

with open('ppo_go1_velocity.yaml') as f:
    cfg = yaml.safe_load(f)

c = cfg['params']['config']
c['num_actors'] = NUM_ENVS
c['minibatch_size'] = 8192
c['max_epochs'] = 1000          # full scale: 10000
c['name'] = 'MJLab_Go1_colab'
c['save_frequency'] = 250
c['device'] = 'cuda:0'
cv = c.get('central_value_config')
if cv:
    cv['minibatch_size'] = 8192  # asymmetric critic dataset must divide the batch too
print('envs', c['num_actors'], '| minibatch', c['minibatch_size'], '| epochs', c['max_epochs'])
# NOTE: fewer than ~1024 envs risks a 'stander' local optimum on Go1 --
# the policy farms reward without walking. The visualization notebook's
# velocity probe exposes it. More envs = more gait-discovery diversity.

In [ ]:
from rl_games.torch_runner import Runner

runner = Runner()
runner.load(cfg)
runner.run({'train': True})

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

run_dir = sorted(glob.glob('runs/MJLab_Go1_colab*'))[-1]
ea = EventAccumulator(run_dir + '/summaries', size_guidance={'scalars': 0}); ea.Reload()
s = ea.Scalars('rewards/iter')
steps, vals = [x.step for x in s], [x.value for x in s]
plt.figure(figsize=(8, 4))
plt.plot(steps, vals)
plt.xlabel('epoch'); plt.ylabel('episode reward'); plt.grid(alpha=0.3)
plt.title(f'Go1 velocity, {NUM_ENVS} envs — final {np.mean(vals[-20:]):.1f}')
plt.show()
CHECKPOINT = run_dir + '/nn/MJLab_Go1_colab.pth'
print('checkpoint:', CHECKPOINT)

## Full-scale training

Locally: `python runner.py --train --file rl_games/configs/mjlab/ppo_go1_velocity.yaml`
(4096 envs, 10k epochs; multi-GPU via torchrun with envs-per-GPU kept constant).
Benchmarks and more tasks (G1 humanoid, cube lifting, dexterous hands): `docs/MJLAB.md`.